In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from utils import preprocess_data, get_features_and_target
from classification_utils import (
    train_and_compare_classification,
    print_results_table,
    print_confusion_matrix
)

In [2]:
#Подготовка данных
df = preprocess_data('data.xlsx')
# Используем порог 8 вместо медианы
X, y = get_features_and_target(df, 'SI',
                                task_type='classification',
                                threshold=8)

print(f"\nЦелевая переменная: SI > 8 (фиксированный порог 8)")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nБаланс классов:")
print(f"  Класс 0 (SI <= 8): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Класс 1 (SI > 8):  {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")


Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: SI > 8 (фиксированный порог 8)
Размер X: (1001, 144), размер y: (1001,)

Баланс классов:
  Класс 0 (SI <= 8): 644 (64.3%)
  Класс 1 (SI > 8):  357 (35.7%)


In [3]:
#Обучение и сравнение моделей
results_df, best_models, splits = train_and_compare_classification(
    X, y,
    target_name='SI > 8',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [5]:
#Вывод таблицы
print_results_table(results_df, target_name='SI > 8')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_classification_SI_8.csv', index=False)

             Model    CV F1  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC_AUC
               SVC 0.559521       0.677741        0.545455     0.560748 0.552995      0.696430
               KNN 0.580758       0.674419        0.540541     0.560748 0.550459      0.694431
  GradientBoosting 0.557991       0.700997        0.595506     0.495327 0.540816      0.712954
LogisticRegression 0.539458       0.681063        0.555556     0.514019 0.533981      0.676486
      RandomForest 0.572706       0.697674        0.593023     0.476636 0.528497      0.710979
      DecisionTree 0.542306       0.617940        0.467742     0.542056 0.502165      0.611306

Лучшая модель: SVC (Test F1 = 0.5530)


In [7]:
#визуализация и анализ лучшей модели
# Сравнение моделей по F1-score
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test F1', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test F1'], color='steelblue', edgecolor='black')
plt.xlabel('F1-score на тестовой выборке')
plt.title('Сравнение моделей классификации по F1-score (SI > 8)')
plt.tight_layout()
plt.savefig('results/comparison_classification_SI_8.png',
            dpi=100, bbox_inches='tight')
plt.close()

# Анализ лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

y_pred_test = best_model.predict(X_test)

print_confusion_matrix(y_test, y_pred_test, model_name=best_model_name)
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['SI <= 8', 'SI > 8'],
            yticklabels=['SI <= 8', 'SI > 8'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title(f'Матрица ошибок: {best_model_name}\n(SI > 8)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_SI_8.png', dpi=100, bbox_inches='tight')
plt.close()


TN (правильно отрицательные): 144
FP (ложные срабатывания):     50
FN (пропущенные положительные): 47
TP (правильно положительные): 60
              precision    recall  f1-score   support

           0       0.75      0.74      0.75       194
           1       0.55      0.56      0.55       107

    accuracy                           0.68       301
   macro avg       0.65      0.65      0.65       301
weighted avg       0.68      0.68      0.68       301



In [8]:
#Важность признаков
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\nТоп-15 важных признаков для {best_model_name}")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для классификации SI > 8\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_SI_8.png',
                dpi=100, bbox_inches='tight')
    plt.close()
